# Expression expansion

In [2]:
import polars as pl

In [6]:
df = pl.DataFrame(
    { 
        "ticker": ["AAPL", "NVDA", "MSFT", "GOOG", "AMZN"],
        "company_name": ["Apple", "NVIDIA", "Microsoft", "Alphabet (Google)", "Amazon"],
        "price": [229.9, 138.93, 420.56, 166.41, 188.4],
        "day_high": [231.31, 139.6, 424.04, 167.62, 189.83],
        "day_low": [228.6, 136.3, 417.52, 164.78, 188.44],
        "year_high": [237.23, 140.76, 468.35, 193.31, 201.2],
        "year_low": [164.08, 39.23, 324.39, 121.46, 118.35],
    }
)

df

ticker,company_name,price,day_high,day_low,year_high,year_low
str,str,f64,f64,f64,f64,f64
"""AAPL""","""Apple""",229.9,231.31,228.6,237.23,164.08
"""NVDA""","""NVIDIA""",138.93,139.6,136.3,140.76,39.23
"""MSFT""","""Microsoft""",420.56,424.04,417.52,468.35,324.39
"""GOOG""","""Alphabet (Google)""",166.41,167.62,164.78,193.31,121.46
"""AMZN""","""Amazon""",188.4,189.83,188.44,201.2,118.35


## Col

In [5]:
eur_usd_rate = 1.09

result = df.with_columns(
    (
        pl.col(
            "price",
            "day_high",
            "day_low",
            "year_high",
            "year_low",
        )
        / eur_usd_rate
    ).round(2)
)

result

ticker,company_name,price,day_high,day_low,year_high,year_low
str,str,f64,f64,f64,f64,f64
"""AAPL""","""Apple""",210.92,212.21,209.72,217.64,150.53
"""NVDA""","""NVIDIA""",127.46,128.07,125.05,129.14,35.99
"""MSFT""","""Microsoft""",385.83,389.03,383.05,429.68,297.61
"""GOOG""","""Alphabet (Google)""",152.67,153.78,151.17,177.35,111.43
"""AMZN""","""Amazon""",172.84,174.16,172.88,184.59,108.58


In [7]:
exprs = [
    (pl.col("price") / eur_usd_rate).round(2),
    (pl.col("day_high") / eur_usd_rate).round(2),
    (pl.col("day_low") / eur_usd_rate).round(2),
    (pl.col("year_high") / eur_usd_rate).round(2),
    (pl.col("year_low") / eur_usd_rate).round(2),
]


result2 = df.with_columns(exprs)
result2

ticker,company_name,price,day_high,day_low,year_high,year_low
str,str,f64,f64,f64,f64,f64
"""AAPL""","""Apple""",210.92,212.21,209.72,217.64,150.53
"""NVDA""","""NVIDIA""",127.46,128.07,125.05,129.14,35.99
"""MSFT""","""Microsoft""",385.83,389.03,383.05,429.68,297.61
"""GOOG""","""Alphabet (Google)""",152.67,153.78,151.17,177.35,111.43
"""AMZN""","""Amazon""",172.84,174.16,172.88,184.59,108.58


## Expansion by data type

In [8]:
result = df.with_columns((pl.col(pl.Float64) / eur_usd_rate).round(2))
result

ticker,company_name,price,day_high,day_low,year_high,year_low
str,str,f64,f64,f64,f64,f64
"""AAPL""","""Apple""",210.92,212.21,209.72,217.64,150.53
"""NVDA""","""NVIDIA""",127.46,128.07,125.05,129.14,35.99
"""MSFT""","""Microsoft""",385.83,389.03,383.05,429.68,297.61
"""GOOG""","""Alphabet (Google)""",152.67,153.78,151.17,177.35,111.43
"""AMZN""","""Amazon""",172.84,174.16,172.88,184.59,108.58


## Expansion by pattern matching

In [9]:
result = df.select(pl.col("ticker", "^.*_high$", "^.*_low$"))
result

ticker,day_high,day_low,year_high,year_low
str,f64,f64,f64,f64
"""AAPL""",231.31,228.6,237.23,164.08
"""NVDA""",139.6,136.3,140.76,39.23
"""MSFT""",424.04,417.52,468.35,324.39
"""GOOG""",167.62,164.78,193.31,121.46
"""AMZN""",189.83,188.44,201.2,118.35


# Selection and exclusion

## Select all columns

In [12]:
result = df.select(pl.all())
result

ticker,company_name,price,day_high,day_low,year_high,year_low
str,str,f64,f64,f64,f64,f64
"""AAPL""","""Apple""",229.9,231.31,228.6,237.23,164.08
"""NVDA""","""NVIDIA""",138.93,139.6,136.3,140.76,39.23
"""MSFT""","""Microsoft""",420.56,424.04,417.52,468.35,324.39
"""GOOG""","""Alphabet (Google)""",166.41,167.62,164.78,193.31,121.46
"""AMZN""","""Amazon""",188.4,189.83,188.44,201.2,118.35


## Select by data type

In [27]:
result = df.select(pl.all().exclude(pl.Utf8))
result

price,day_high,day_low,year_high,year_low
f64,f64,f64,f64,f64
229.9,231.31,228.6,237.23,164.08
138.93,139.6,136.3,140.76,39.23
420.56,424.04,417.52,468.35,324.39
166.41,167.62,164.78,193.31,121.46
188.4,189.83,188.44,201.2,118.35


## Exclude columns by pattern matching

In [14]:
result = df.select(pl.all().exclude("^day_.*$"))
result

ticker,company_name,price,year_high,year_low
str,str,f64,f64,f64
"""AAPL""","""Apple""",229.9,237.23,164.08
"""NVDA""","""NVIDIA""",138.93,140.76,39.23
"""MSFT""","""Microsoft""",420.56,468.35,324.39
"""GOOG""","""Alphabet (Google)""",166.41,193.31,121.46
"""AMZN""","""Amazon""",188.4,201.2,118.35


## Swap one column to the front

In [18]:
result = df.select(pl.col("company_name"), pl.all().exclude("company_name"))
result

company_name,ticker,price,day_high,day_low,year_high,year_low
str,str,f64,f64,f64,f64,f64
"""Apple""","""AAPL""",229.9,231.31,228.6,237.23,164.08
"""NVIDIA""","""NVDA""",138.93,139.6,136.3,140.76,39.23
"""Microsoft""","""MSFT""",420.56,424.04,417.52,468.35,324.39
"""Alphabet (Google)""","""GOOG""",166.41,167.62,164.78,193.31,121.46
"""Amazon""","""AMZN""",188.4,189.83,188.44,201.2,118.35


# Renaming

## Renaming columns

In [20]:
gbp_usd_rate = 1.31 

result = df.select(
    (pl.col("price") / gbp_usd_rate).alias("price (GBP)"),
    (pl.col("price") / eur_usd_rate).alias("price (EUR)"),
)

result

price (GBP),price (EUR)
f64,f64
175.496183,210.917431
106.053435,127.458716
321.038168,385.834862
127.030534,152.669725
143.816794,172.844037


## Prefixing and suffixing column names

In [23]:
result = df.select(
    (pl.col("^year_.*$") / eur_usd_rate).name.prefix("in_eur_"),
    (pl.col("^day_.*$") / gbp_usd_rate).name.suffix("_gbp"),
)

result

in_eur_year_high,in_eur_year_low,day_high_gbp,day_low_gbp
f64,f64,f64,f64
217.642202,150.53211,176.572519,174.503817
129.137615,35.990826,106.564885,104.045802
429.678899,297.605505,323.694656,318.717557
177.348624,111.431193,127.954198,125.78626
184.587156,108.577982,144.908397,143.847328


## Dynamic name replacement

In [31]:
result = df.select(pl.all().name.to_uppercase())
result

TICKER,COMPANY_NAME,PRICE,DAY_HIGH,DAY_LOW,YEAR_HIGH,YEAR_LOW
str,str,f64,f64,f64,f64,f64
"""AAPL""","""Apple""",229.9,231.31,228.6,237.23,164.08
"""NVDA""","""NVIDIA""",138.93,139.6,136.3,140.76,39.23
"""MSFT""","""Microsoft""",420.56,424.04,417.52,468.35,324.39
"""GOOG""","""Alphabet (Google)""",166.41,167.62,164.78,193.31,121.46
"""AMZN""","""Amazon""",188.4,189.83,188.44,201.2,118.35
